# ?? Market Candlestick & AI Confluence Scanner (yfinance-ta-patterns)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eminsk/yfinance-ta-patterns/blob/main/notebooks/yfinance_ta_patterns_quickstart.ipynb)
[![PyPI](https://img.shields.io/pypi/v/yfinance-ta-patterns?color=blue)](https://pypi.org/project/yfinance-ta-patterns/)
[![Conda Version](https://img.shields.io/conda/vn/conda-forge/yfinance-ta-patterns.svg)](https://anaconda.org/conda-forge/yfinance-ta-patterns)
[![Ubuntu / Debian PPA](https://img.shields.io/badge/Ubuntu%20%2F%20Debian-APT%20PPA-E95420?logo=ubuntu&logoColor=white)](https://eminsk.github.io/ppa/)
[![License: MIT](https://img.shields.io/badge/License-MIT-green.svg)](https://opensource.org/licenses/MIT)

Welcome to the interactive quickstart notebook for **yfinance-ta-patterns**!

yfinance-ta-patterns is an institutional-grade quantitative Python library and CLI tool designed for automated candlestick pattern detection, multi-factor **AI Confluence Scoring** (EMA 20/50/200 trend alignment, Wilder's RSI, RVOL, ATR volatility), automated trade setup generation (Entry, Stop Loss, Take Profit 1.5R/3.0R), and unbiased next-open backtesting with transaction costs and Sharpe ratio analysis.

Works out of the box with **zero C compilation requirements** thanks to its built-in vectorized pure-NumPy fallback engine, with optional native C TA-Lib hardware acceleration.

## 1. ? Quick Installation

Install yfinance-ta-patterns conflict-free directly from PyPI:

In [ ]:
# Install yfinance-ta-patterns cleanly from PyPI (preserves Colab numpy/pandas):
!pip install -q yfinance-ta-patterns


## 2. ?? Multi-Asset Data Loading & Normalization

MarketDataLoader fetches and normalizes historical candles for equities, ETFs, crypto pairs (BTC-USD), forex pairs (EURUSD=X), and commodities (GC=F).

In [ ]:
from yfinance_ta_patterns import MarketDataLoader

# Fetch historical daily OHLCV candles for NVIDIA (NVDA):
loader = MarketDataLoader("NVDA", interval="1d", period="1y", repair=False)
df = loader.get_data()

print(f"Asset Symbol:       NVDA")
print(f"Total Bars:         {len(df)}")
print(f"Date Range:         {df.index[0].date()} -> {df.index[-1].date()}")
print(f"Latest Close Price: ")
df.tail(5)


## 3. ?? Candlestick Pattern Recognition

PatternAnalyzer scans historical OHLC data across classic candlestick patterns (Hammer, Engulfing, Morning Star, Shooting Star, Doji, etc.) and tags exact timestamps with bullish (+100) or bearish (-100) signal strengths.

In [ ]:
from yfinance_ta_patterns import PatternAnalyzer
import pandas as pd

analyzer = PatternAnalyzer(df)
print(f"Active Pattern Recognition Functions: {len(analyzer.pattern_functions)}")

detected_events = []
for pattern in analyzer.pattern_functions:
    try:
        signals = analyzer.get_signals(pattern)
        active = signals[signals != 0]
        for ts, val in active.items():
            detected_events.append({
                "Date": ts.date(),
                "Pattern": pattern,
                "Signal": "BULLISH (BUY)" if val > 0 else "BEARISH (SELL)",
                "Strength": abs(int(val)),
                "Close": round(float(df.loc[ts, "Close"]), 2)
            })
    except NotImplementedError:
        continue

df_events = pd.DataFrame(detected_events)
if not df_events.empty:
    df_events = df_events.sort_values(by="Date", ascending=False).reset_index(drop=True)
    print(f"\nTotal candlestick patterns detected: {len(df_events)}")
    display(df_events.head(10))
else:
    print("No patterns found in the selected lookback range.")


## 4. ?? AI & Quant Confluence Scoring

AIPatternScorer enriches raw technical signals with multi-factor confluence analysis:
- **Trend Regime**: Triple EMA alignment (20, 50, 200 EMA).
- **Momentum Regime**: Canonical Wilder's RSI (14) overbought/oversold and divergence detection.
- **Volume Surge**: Relative Volume (RVOL) confirmation.
- **Volatility**: Wilder's ATR (14) expansion and candle body dominance.

It computes an objective **Confluence Score (0.0 to 1.0)**, assigns a **Signal Grade** (EXCELLENT, STRONG, MODERATE, WEAK), and generates an automated **Trade Setup** (Entry, Stop Loss, Take Profit 1.5R / 3.0R).

In [ ]:
from yfinance_ta_patterns import AIPatternScorer

scorer = AIPatternScorer(df)

# Scan all signals occurring in the last 30 trading bars with confidence >= 45%:
active_signals = scorer.score_all_active(min_confidence=0.45, lookback_bars=30)
print(f"Total Confluent Signals Found: {len(active_signals)}\n")

for sig in active_signals[:5]:
    setup = sig.trade_setup
    print(f"[{sig.grade.name}] {sig.pattern_name} at {sig.timestamp.date()} | AI Confluence: {sig.confidence_score * 100:.1f}/100")
    if setup:
        print(f"  -> Action: {setup.direction} @ ")
        print(f"  -> Stop Loss:     ")
        print(f"  -> Target 1 (1.5R): ")
        print(f"  -> Target 2 (3.0R): ")
    for factor in sig.confluence_factors:
        print(f"  + {factor}")
    print("-" * 65)


## 5. ?? Executive Market Brief & AI Agent Prompt Generation

AIMarketAnalyst transforms complex pattern detections into an executive Markdown brief, as well as zero-shot prompts tailored for LLM agents (ChatGPT, Claude, Gemini, DeepSeek):

In [ ]:
from yfinance_ta_patterns import AIMarketAnalyst
from IPython.display import Markdown

analyst = AIMarketAnalyst(df, symbol="NVDA")

# 1. Display executive formatted brief:
brief = analyst.generate_brief()
display(Markdown(brief))

# 2. Generate structured prompt for AI Agents:
llm_prompt = analyst.to_llm_prompt()
print("Sample AI Agent Prompt Header (First 250 characters):")
print(llm_prompt[:250] + "...")


## 6. ?? Quantitative Historical Backtesting

PatternRankingTester runs backtests with **zero lookahead bias** (unbiased next-open execution Open[i+1]), accounting for transaction commissions, slippage, holding periods, and periodic Sharpe ratios.

In [ ]:
from yfinance_ta_patterns import PatternRankingTester

tester = PatternRankingTester(
    data=df,
    symbol="NVDA",
    account_currency="USD",
    initial_capital=10000.0,
    position_size=1000.0,
    execution="next_open",  # Unbiased next-open execution
    holding_period=5,        # 5-day holding period exit
    min_signals=2,           # Filter out random one-offs
    commission=1.0,          # .00 fixed commission per trade
)

results = tester.test_all_patterns(sort_by="composite")
print(f"Backtest successfully evaluated {len(results)} pattern strategies:\n")

rows = []
for r in results:
    rows.append({
        "Pattern": r.pattern_name,
        "Trades": r.total_trades,
        "Win Rate (%)": f"{r.win_rate:.1f}%",
        "Total PnL ($)": f"",
        "Profit Factor": f"{r.profit_factor:.2f}" if r.profit_factor else "N/A",
        "Sharpe Ratio": f"{r.sharpe_ratio:.2f}" if r.sharpe_ratio else "N/A",
        "Max Drawdown": f""
    })

df_backtest = pd.DataFrame(rows)
display(df_backtest)


## 7. ? Command-Line Interface (CLI)

yfinance-ta-patterns includes the yftp CLI command for instant shell scanning:

In [ ]:
# Execute a live AI-confluence scan from the command line:
!yftp --symbol NVDA --timeframe 1d --period 3mo --all-patterns --ai


## 8. ?? Summary & Official Resources

- **GitHub Repository**: [eminsk/yfinance-ta-patterns](https://github.com/eminsk/yfinance-ta-patterns)
- **PyPI Package**: [pip install yfinance-ta-patterns](https://pypi.org/project/yfinance-ta-patterns/)
- **Conda-Forge**: [conda install -c conda-forge yfinance-ta-patterns](https://anaconda.org/conda-forge/yfinance-ta-patterns)
- **Ubuntu / Debian PPA**: [https://eminsk.github.io/ppa/](https://eminsk.github.io/ppa/)
- **Documentation**: [https://github.com/eminsk/yfinance-ta-patterns#readme](https://github.com/eminsk/yfinance-ta-patterns#readme)
